# Stage 2 — Governed Autonomous Experimentation: Overview and Demo

**Paper framing:** *Auditable Autonomous Experimentation with Bounded Execution and Explicit Keep/Discard Control*

This notebook walks through the complete Stage 2 framework: what was built, why each layer exists, and live demonstrations of every component. Run cells top to bottom.

---

## What Stage 2 is

Stage 1 demonstrated a working governed campaign on a real ML node (ResNet-trigger, +0.0076 val_auc over baseline). Stage 2 formalises that prototype into a research artifact:

- Explicit trial lifecycle (initialized → proposed → executed → evaluated → kept/discarded/failed_invalid)
- Append-only JSONL trial ledger — nothing is overwritten
- Formal node contracts (editable paths, frozen paths, metric parser, acceptance rule)
- Three memory modes for an ablation experiment
- Three interchangeable manager backends (baseline, prompt, LangGraph)
- Paper-ready metric tables and figure CSVs

## What Stage 2 is not

- Not a general autonomous scientist
- Not a LangChain demo — LangGraph is one optional manager backend only
- Not proof of scientific discovery
- Not a wrapper around a coding agent

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0] if Path().resolve().name == 'notebooks' else Path().resolve()
SRC = ROOT / 'src'
VENV_SITE = ROOT / '.venv' / 'lib'

for p in [str(SRC)]:
    if p not in sys.path:
        sys.path.insert(0, p)

if VENV_SITE.exists():
    for _p in sorted(VENV_SITE.iterdir()):
        _site = _p / 'site-packages'
        if _site.exists() and str(_site) not in sys.path:
            sys.path.insert(0, str(_site))

print('ROOT:', ROOT)
print('SRC on path:', SRC.exists())
print('venv on path:', VENV_SITE.exists())

---
## 1. The Node Contract

Every experiment target must declare a formal `NodeSpec`: which files the worker may edit, which are frozen, how to run the experiment, what metric to parse, and how to decide keep vs discard. This is the boundary that makes the framework auditable.

In [ ]:
from autoresearch.nodes.registry import load_registered_node
import json

node_spec = load_registered_node('resnet_trigger', repo_root=ROOT)

print('Node:', node_spec.name)
print('Description:', node_spec.description)
print()
print('Editable paths (worker may touch):', node_spec.editable_paths)
print('Frozen paths  (worker may NOT touch):', node_spec.frozen_paths)
print()
print('Run command  :', node_spec.run_command)
print('Metric       :', node_spec.metric_name, '|', node_spec.metric_direction)
print('Acceptance   :', node_spec.acceptance_rule)
print('Default budget:', node_spec.default_budget)

The node spec is also inspectable from the CLI:
```bash
python3 scripts/inspect_node.py --node resnet_trigger
```

---
## 2. The Trial Lifecycle

The control plane enforces a formal state machine. A trial cannot be decided before the metric is parsed; it cannot execute before scope validation. Invalid state transitions raise an error.

In [ ]:
from autoresearch.control_plane.state_machine import TrialState, transition

# Show valid forward path
path = [TrialState.INITIALIZED]
for next_state in [
    TrialState.PROPOSED,
    TrialState.PATCH_GENERATED,
    TrialState.SCOPE_VALIDATED,
    TrialState.EXECUTED,
    TrialState.METRIC_PARSED,
    TrialState.EVALUATED,
    TrialState.KEPT,
]:
    path.append(transition(path[-1], next_state))

print('Valid lifecycle path:')
for i, s in enumerate(path):
    print(f'  {i}: {s.value}')

# Show invalid transition is rejected
print()
try:
    transition(TrialState.INITIALIZED, TrialState.KEPT)  # skip everything
    print('ERROR: should have been rejected')
except Exception as e:
    print('Invalid jump rejected:', str(e)[:80])

---
## 3. Editable-Scope Validation

Before any trial executes, the control plane checks that the worker only touched files declared as editable in the node spec. Scope violations become `failed_invalid` without running the experiment.

In [ ]:
from autoresearch.control_plane.permissions import validate_edit_scope

# Valid: only train.py
valid = validate_edit_scope(('train.py',), node_spec)
print('train.py only    →', 'VALID' if valid.valid else 'INVALID', valid.violations)

# Invalid: touched prepare.py (frozen)
invalid = validate_edit_scope(('train.py', 'prepare.py'), node_spec)
print('+ prepare.py     →', 'VALID' if invalid.valid else 'INVALID', invalid.violations)

# Invalid: touched a data file
data = validate_edit_scope(('train.py', 'data/train.h5'), node_spec)
print('+ data/train.h5  →', 'VALID' if data.valid else 'INVALID', data.violations)

---
## 4. The Decision Module

The Stage 2 control plane owns keep/discard. The worker returns a metric; the decision module compares it against the current best and emits a `DecisionResult`. This is deterministic and unit-testable — the LLM cannot override it.

In [ ]:
from autoresearch.control_plane.decision import decide_trial
from autoresearch.memory.schemas import ValidityStatus

# Case 1: improvement
d = decide_trial(
    validity_status=ValidityStatus.VALID,
    candidate_metric=0.787,
    current_best_metric=0.782,
    metric_direction='maximize',
)
print('Improvement →', d.decision.value, '| delta:', d.delta_vs_best)

# Case 2: degraded
d = decide_trial(
    validity_status=ValidityStatus.VALID,
    candidate_metric=0.775,
    current_best_metric=0.782,
    metric_direction='maximize',
)
print('Degraded    →', d.decision.value, '| delta:', d.delta_vs_best)

# Case 3: invalid (scope violation, metric missing, runtime crash)
from autoresearch.memory.schemas import FailureCategory
d = decide_trial(
    validity_status=ValidityStatus.INVALID,
    candidate_metric=None,
    current_best_metric=0.782,
    metric_direction='maximize',
    failure_category=FailureCategory.INVALID_EDIT_SCOPE,
)
print('Invalid     →', d.decision.value, '|', d.failure_category.value)

---
## 5. Budget Enforcement

The `BudgetState` is an immutable frozen dataclass — consuming it returns a new state. The campaign runner can never silently over-run the budget.

In [ ]:
from autoresearch.control_plane.budget import BudgetState, BudgetExceededError

budget = BudgetState(total_trials=3)
print('Start:', budget.to_dict())

budget = budget.consume_one()
budget = budget.consume_one()
budget = budget.consume_one()
print('After 3:', budget.to_dict())

try:
    budget.consume_one()
except BudgetExceededError as e:
    print('Over-run blocked:', e)

---
## 6. The Three Manager Backends

All three managers implement the same interface: `propose_next_trial(status, memory_context, node_spec) → ManagerProposal`. The control plane treats them identically.

In [ ]:
from autoresearch.manager.base import ManagerStatus
from autoresearch.manager.baseline_manager import BaselineManager
from autoresearch.manager.prompt_manager import PromptManager
from autoresearch.memory.summarizer import MemoryMode, build_memory_context

status = ManagerStatus(
    campaign_id='demo',
    budget_index=2,
    current_best_metric=0.782,
    metric_name='val_auc',
    metric_direction='maximize',
)
ctx_none = build_memory_context([], MemoryMode.NONE, node_spec, budget_index=2)

# BaselineManager — heuristic, no LLM
baseline = BaselineManager()
p = baseline.propose_next_trial(status, ctx_none, node_spec)
print('BaselineManager')
print('  mode    :', p.manager_mode)
print('  summary :', p.proposal_summary)
print('  files   :', p.target_files)
print()

# PromptManager — memory-aware, text construction, no LLM call
prompt_mgr = PromptManager()
ctx_rationale = build_memory_context([], MemoryMode.APPEND_ONLY_SUMMARY_WITH_RATIONALE, node_spec, budget_index=2)
p = prompt_mgr.propose_next_trial(status, ctx_rationale, node_spec)
print('PromptManager')
print('  mode    :', p.manager_mode)
print('  summary :', p.proposal_summary)
print('  objective (first 80 chars):', p.objective[:80])

In [ ]:
# LangGraphManager — three-node planning graph, LLM injected for testability
from langchain_core.language_models import FakeListChatModel
from autoresearch.manager.langgraph_manager import LangGraphManager
import json

stub_response = json.dumps({
    'summary': 'reduce-lr-5e-4',
    'rationale': 'Smaller learning rate typically improves late-stage convergence.',
    'objective': 'In train.py, change learning_rate from 1e-3 to 5e-4. Edit only train.py. Make no other changes.',
})
fake_llm = FakeListChatModel(responses=[stub_response])
lg_manager = LangGraphManager(llm=fake_llm)

p = lg_manager.propose_next_trial(status, ctx_rationale, node_spec)
print('LangGraphManager')
print('  mode     :', p.manager_mode)
print('  summary  :', p.proposal_summary)
print('  rationale:', p.proposal_rationale)
print('  objective:', p.objective)
print('  files    :', p.target_files)

---
## 7. Memory Modes

The memory ablation tests whether the manager produces better proposals when it receives prior trial summaries. Three modes build increasingly rich contexts.

In [ ]:
from autoresearch.memory.summarizer import build_memory_context, MemoryMode
from autoresearch.memory.schemas import TrialRecord, TrialDecision, ExecutionStatus, ValidityStatus
from autoresearch.memory.provenance import build_trial_provenance

# Build a synthetic prior trial to populate memory
prior = TrialRecord(
    trial_id='demo-trial-001',
    campaign_id='demo',
    node_id='resnet_trigger',
    budget_index=1,
    timestamp_start='2026-04-29T10:00:00Z',
    timestamp_end='2026-04-29T10:18:00Z',
    manager_mode='prompt_manager',
    worker_mode='claw_style_worker',
    memory_mode='none',
    proposal_summary='reduce-lr-5e-4',
    proposal_rationale='smaller lr improves convergence',
    targeted_files=('train.py',),
    patch_ref='experiments/artifacts/trial-001/generated_packet.json',
    git_commit_before='abc123',
    git_commit_after='def456',
    execution_status=ExecutionStatus.SUCCESS,
    validity_status=ValidityStatus.VALID,
    failure_category=None,
    raw_log_ref='experiments/artifacts/trial-001/run.log',
    parsed_metrics={'val_auc': 0.7826},
    current_best_before=0.7799,
    delta_vs_best=0.0027,
    decision=TrialDecision.KEPT,
    decision_rationale='Candidate improved maximize objective by 0.002700.',
    wall_clock_seconds=1082.0,
    cumulative_budget_consumed=1,
    provenance=build_trial_provenance('demo-trial-001'),
)

for mode in MemoryMode:
    ctx = build_memory_context([prior], mode, node_spec, budget_index=2)
    print(f'\n--- {mode.value} ---')
    print(f'  raw chars: {ctx.raw_memory_chars}  compressed: {ctx.compressed_chars}  ratio: {ctx.compression_ratio:.2f}')
    print('  context preview:', ctx.context_text[:120].replace('\n', ' | '))

---
## 8. Repeated-Bad-Idea Detection

The `append_only_summary_with_rationale` mode includes warnings when the manager keeps proposing changes similar to previously failed or discarded trials. This is computed by `similarity.py` and appears in the manager context and in the paper metrics.

In [ ]:
from autoresearch.memory.similarity import compute_repeated_bad_stats
from autoresearch.memory.schemas import FailureCategory

# Build two discarded trials with similar proposals
def _make_trial(tid, summary, decision, failure=None, metric=0.775):
    return TrialRecord(
        trial_id=tid, campaign_id='demo', node_id='resnet_trigger',
        budget_index=int(tid[-1]),
        timestamp_start='2026-04-29T10:00:00Z', timestamp_end='2026-04-29T10:18:00Z',
        manager_mode='prompt_manager', worker_mode='claw_style_worker', memory_mode='none',
        proposal_summary=summary, proposal_rationale='',
        targeted_files=('train.py',), patch_ref='', git_commit_before='', git_commit_after='',
        execution_status=ExecutionStatus.SUCCESS, validity_status=ValidityStatus.VALID,
        failure_category=failure, raw_log_ref='',
        parsed_metrics={'val_auc': metric} if metric else {},
        current_best_before=0.782, delta_vs_best=metric - 0.782 if metric else None,
        decision=decision, decision_rationale='',
        wall_clock_seconds=100.0, cumulative_budget_consumed=int(tid[-1]),
        provenance=build_trial_provenance(tid),
    )

records = [
    _make_trial('demo-trial-1', 'increase-lr-1e-2', TrialDecision.DISCARDED, failure=FailureCategory.DEGRADED_METRIC, metric=0.771),
    _make_trial('demo-trial-2', 'increase-lr-2e-2', TrialDecision.DISCARDED, failure=FailureCategory.DEGRADED_METRIC, metric=0.769),
    _make_trial('demo-trial-3', 'reduce-dropout-0.1', TrialDecision.KEPT, metric=0.784),
    _make_trial('demo-trial-4', 'increase-lr-3e-2', TrialDecision.DISCARDED, failure=FailureCategory.DEGRADED_METRIC, metric=0.768),
]

stats = compute_repeated_bad_stats(records)
print('Repeated bad proposals detected:', stats.repeated_bad_count)
print('Flagged trial IDs             :', stats.flagged_trial_ids)
print('Repeated degraded count       :', stats.repeated_degraded_count)
print('Repeated invalid count        :', stats.repeated_invalid_count)

---
## 9. The ClawWorker Packet Generation

This is the key fix from Stage 2 development. `ClawWorker.run_trial()` no longer uses a static packet file. It generates a packet from the `ManagerProposal`, so the manager's objective (and memory context) actually drives what the worker does.

In [ ]:
from autoresearch.manager.base import ManagerProposal
from autoresearch.worker.claw_worker import _packet_from_proposal

proposal = ManagerProposal(
    manager_mode='langgraph_manager',
    proposal_summary='reduce-lr-5e-4',
    proposal_rationale='Smaller lr improved convergence in similar tasks.',
    target_files=('train.py',),
    objective='In train.py, change learning_rate from 1e-3 to 5e-4. Edit only train.py. Do not run the experiment.',
)

packet = _packet_from_proposal(proposal, node_spec, budget_index=3)
print('Generated packet (what gets sent to the worker):')
for k, v in packet.items():
    print(f'  {k}: {str(v)[:80]}')

print()
print('Key invariant: objective comes from the proposal, not a static file:')
print('  packet["objective"] == proposal.objective →', packet['objective'] == proposal.objective)

---
## 10. Pending-Trial Guard

Before calling the worker, the control plane acquires a lock file. If the process crashes mid-trial, the guard remains and the next run raises `PendingTrialError` instead of silently overwriting state.

In [ ]:
import tempfile
from autoresearch.control_plane.campaign import _acquire_pending, _release_pending, PendingTrialError

with tempfile.TemporaryDirectory() as d:
    records_path = Path(d) / 'demo_trials.jsonl'

    guard = _acquire_pending(records_path, 'demo-trial-001')
    print('Guard acquired:', guard.exists())
    print('Guard file    :', guard.name)

    try:
        _acquire_pending(records_path, 'demo-trial-002')
    except PendingTrialError as e:
        print('Second trial blocked:', str(e)[:70])

    _release_pending(guard)
    print('Guard released:', not guard.exists())

    guard2 = _acquire_pending(records_path, 'demo-trial-002')
    print('New trial starts cleanly:', guard2.exists())
    _release_pending(guard2)

---
## 11. Full Dry-Run Campaign: baseline_manager

A complete 5-trial campaign with the baseline manager and `memory_mode=none`. Uses `DryRunWorker` (synthetic metrics, no Ollama). Every step of the control-plane path executes; only the worker is mocked.

In [ ]:
from autoresearch.control_plane.campaign import run_dry_campaign
from autoresearch.memory.append_store import TrialAppendStore

with tempfile.TemporaryDirectory() as d:
    records_path = Path(d) / 'baseline_demo_trials.jsonl'

    result = run_dry_campaign(
        node_spec=node_spec,
        campaign_id='baseline_demo',
        budget=5,
        manager_mode='baseline_manager',
        memory_mode='none',
        records_path=records_path,
    )
    print('Campaign result:', result.to_dict())

    records = TrialAppendStore(records_path).read_all()
    print(f'\n{len(records)} TrialRecords written:')
    for r in records:
        print(f'  {r.trial_id}  budget={r.budget_index}  decision={r.decision.value}  val_auc={r.parsed_metrics.get("val_auc", "N/A")}')

---
## 12. Full Dry-Run Campaign: langgraph_manager

Same campaign path with `langgraph_manager`. The LangGraph `prepare_context → generate_proposal → validate_proposal` graph fires for each trial. Stub LLM so no Ollama needed.

In [ ]:
from autoresearch.manager.langgraph_manager import LangGraphManager
from langchain_core.language_models import FakeListChatModel
import autoresearch.control_plane.campaign as campaign_mod

proposals = [
    {'summary': 'reduce-lr-5e-4',   'rationale': 'lr sensitivity search', 'objective': 'Change lr to 5e-4.'},
    {'summary': 'add-dropout-0.3',  'rationale': 'regularisation',        'objective': 'Add dropout 0.3.'},
    {'summary': 'reduce-wd-1e-5',   'rationale': 'less weight decay',     'objective': 'Set weight_decay=1e-5.'},
]
fake_llm = FakeListChatModel(responses=[json.dumps(p) for p in proposals])

with tempfile.TemporaryDirectory() as d:
    records_path = Path(d) / 'lg_demo_trials.jsonl'

    result = run_dry_campaign(
        node_spec=node_spec,
        campaign_id='lg_demo',
        budget=3,
        manager_mode='langgraph_manager',
        memory_mode='append_only_summary_with_rationale',
        records_path=records_path,
        manager_llm=fake_llm,
    )

    records = TrialAppendStore(records_path).read_all()
    print('LangGraph campaign:')
    for r in records:
        print(f'  {r.trial_id}  manager={r.manager_mode}  summary={r.proposal_summary}  decision={r.decision.value}')

---
## 13. Memory Ablation: Three Modes Side-by-Side

This is the primary KDD experiment. Three equal-budget campaigns run with different memory modes. The dry-run version below shows the correct wiring — real campaigns need Ollama.

In [ ]:
from autoresearch.evaluation.ablations import build_memory_ablation_plan
from autoresearch.evaluation.campaign_summary import load_campaign_summary

MEMORY_MODES = ('none', 'append_only_summary', 'append_only_summary_with_rationale')
BUDGET = 3

all_records = []
summaries = []

with tempfile.TemporaryDirectory() as d:
    for mode in MEMORY_MODES:
        cid = f'ablation_{mode}'
        rp = Path(d) / f'{cid}_trials.jsonl'
        run_dry_campaign(
            node_spec=node_spec, campaign_id=cid, budget=BUDGET,
            manager_mode='baseline_manager', memory_mode=mode, records_path=rp,
        )
        records = TrialAppendStore(rp).read_all()
        all_records.extend(records)
        summary = load_campaign_summary(rp)
        summaries.append((mode, summary.metrics))

    print(f'{'Memory mode':<45} {'kept':>6} {'invalid':>8} {'accept_rate':>12} {'net_gain':>10}')
    print('-' * 85)
    for mode, m in summaries:
        print(f'{mode:<45} {m.kept_count:>6} {m.failed_invalid_count:>8} {m.acceptance_rate:>12.2f} {m.net_gain:>10.4f}')

    # Ablation plan rows (for paper table)
    rows = build_memory_ablation_plan(node_spec=node_spec, records=all_records, budget=BUDGET)
    print(f'\nAblation summary rows:')
    for row in rows:
        print(f'  {row.memory_mode:<45} repeated_bad={row.repeated_bad_count}  compression={row.compression_ratio:.2f}')

---
## 14. Paper Metrics from a Real Ledger

All metrics are computed deterministically from the trial ledger — never from live agent state.

In [ ]:
from autoresearch.evaluation.campaign_summary import load_campaign_summary

# Use the real langgraph_smoke ledger produced earlier
ledger = ROOT / 'experiments' / 'ledgers' / 'langgraph_smoke_trials.jsonl'

if ledger.exists():
    summary = load_campaign_summary(ledger)
    m = summary.metrics
    print('Campaign:', m.campaign_id)
    print(f'  initial metric      : {m.initial_metric:.4f}')
    print(f'  best metric         : {m.best_metric:.4f}')
    print(f'  net gain            : {m.net_gain:.4f}')
    print(f'  kept / discarded / invalid : {m.kept_count} / {m.discarded_count} / {m.failed_invalid_count}')
    print(f'  acceptance rate     : {m.acceptance_rate:.2f}')
    print(f'  complete provenance : {m.complete_provenance_rate:.2f}')
    print(f'  gain per trial      : {m.gain_per_trial:.4f}')
    print(f'  gain per hour       : {m.gain_per_hour:.4f}')
else:
    print('Run the langgraph_smoke campaign first (see cell below)')

---
## 15. Paper Table Export

Scripts write deterministic CSVs from any ledger. These are the direct paper inputs.

In [ ]:
import subprocess, sys

ledger = ROOT / 'experiments' / 'ledgers' / 'langgraph_smoke_trials.jsonl'
if ledger.exists():
    r = subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'summarize_campaign.py'),
         '--campaign-id', 'langgraph_smoke',
         '--records', str(ledger)],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        import json
        d = json.loads(r.stdout)
        print('Tables written:')
        for k, v in d.get('tables', {}).items():
            print(f'  {k}: {v}')
    else:
        print(r.stderr)
else:
    print('No langgraph_smoke ledger found.')

In [ ]:
# Show the governance_metrics.csv
import csv
gov_csv = ROOT / 'paper' / 'tables' / 'governance_metrics.csv'
if gov_csv.exists():
    rows = list(csv.DictReader(gov_csv.open()))
    if rows:
        for k, v in rows[0].items():
            print(f'  {k}: {v}')
else:
    print('Run summarize_campaign.py first.')

---
## 16. LangGraph Manager: Graph Structure

The LangGraph graph is scoped to one task: turn `(status, memory_context, node_spec) → ManagerProposal`. Three nodes, no conditional edges, no loops.

In [ ]:
from langchain_core.language_models import FakeListChatModel
from autoresearch.manager.langgraph_manager import LangGraphManager, _prepare_context, _validate_proposal

# Show the graph node names and edges by inspecting the compiled graph
fake = FakeListChatModel(responses=[json.dumps({'summary': 's', 'rationale': 'r', 'objective': 'o'})])
lg = LangGraphManager(llm=fake)
graph = lg._get_graph()

print('Graph nodes:')
for node in graph.nodes:
    print(f'  {node}')

print()
print('Architecture invariant — LangGraph may only access:')
for allowed in ['ManagerStatus', 'MemoryContext', 'NodeSpec', 'ManagerProposal']:
    print(f'  ✓ {allowed}')
print()
print('LangGraph must never access:')
for forbidden in ['BudgetState', 'TrialRecord', 'TrialAppendStore', 'WorkerResult', 'TrialDecision']:
    print(f'  ✗ {forbidden}')

---
## 17. Inline Smoke Tests

Run all six smoke suites from this notebook to confirm nothing is broken.

In [ ]:
import subprocess, sys

smoke_tests = [
    ('Contracts + lifecycle',        sys.executable, 'notebooks/stage2_development_smoke_tests/stage2_priority_0_1_contracts_smoke.py'),
    ('ResNet node integration',      sys.executable, 'notebooks/stage2_development_smoke_tests/stage2_priority_2_3_integration_smoke.py'),
    ('Memory modes + managers',      sys.executable, 'notebooks/stage2_development_smoke_tests/stage2_priority_4_memory_and_manager_smoke.py'),
    ('System integration',           sys.executable, 'notebooks/stage2_development_smoke_tests/stage2_priority_6_16_system_smoke.py'),
    ('Proposal → TrialRecord chain', sys.executable, 'notebooks/stage2_development_smoke_tests/stage2_proposal_to_trial_smoke.py'),
    ('LangGraph manager',            str(ROOT / '.venv' / 'bin' / 'python'), 'notebooks/stage2_development_smoke_tests/stage2_langgraph_manager_smoke.py'),
]

all_passed = True
for label, python, script in smoke_tests:
    env = {'PYTHONPATH': str(ROOT / 'src')}
    import os
    full_env = {**os.environ, **env}
    r = subprocess.run(
        [python, str(ROOT / script)],
        capture_output=True, text=True, env=full_env
    )
    status = 'PASS' if r.returncode == 0 else 'FAIL'
    if r.returncode != 0:
        all_passed = False
    last_line = (r.stdout + r.stderr).strip().splitlines()[-1] if (r.stdout + r.stderr).strip() else ''
    print(f'[{status}] {label:<35} {last_line}')

print()
print('All smoke tests passed.' if all_passed else 'SOME TESTS FAILED — see output above.')

---
## 18. Reproducing the Main Campaign Command

This is the exact command used to verify the full pipeline. With `--llm-stub` it works without a running Ollama instance.

In [ ]:
import subprocess, json, sys

r = subprocess.run(
    [
        sys.executable,
        str(ROOT / 'scripts' / 'run_campaign.py'),
        '--node', 'resnet_trigger',
        '--campaign-id', 'notebook_demo',
        '--budget', '3',
        '--manager', 'langgraph_manager',
        '--memory-mode', 'append_only_summary_with_rationale',
        '--dry-run',
        '--llm-stub',
    ],
    capture_output=True, text=True,
    env={**__import__('os').environ, 'PYTHONPATH': str(ROOT / 'src')}
)
if r.returncode == 0:
    d = json.loads(r.stdout)
    print('records_written:', d['campaign']['records_written'])
    print('manager_mode in ledger: langgraph_manager ✓') if d['campaign']['records_written'] > 0 else None
    print('\nMetrics:')
    for k in ('kept_count', 'acceptance_rate', 'net_gain', 'complete_provenance_rate'):
        print(f'  {k}: {d["metrics"][k]}')
else:
    print('FAILED:', r.stderr[-500:])

---
## Summary: What Stage 2 builds

| Component | File | Status |
|---|---|---|
| Node spec + registry | `nodes/spec.py`, `nodes/registry.py` | ✅ |
| Trial record schema | `memory/schemas.py` | ✅ |
| State machine | `control_plane/state_machine.py` | ✅ |
| Editable-scope check | `control_plane/permissions.py` | ✅ |
| Decision module | `control_plane/decision.py` | ✅ |
| Budget enforcement | `control_plane/budget.py` | ✅ |
| Append-only ledger | `memory/append_store.py` | ✅ |
| Provenance IDs | `memory/provenance.py` | ✅ |
| Memory modes (3) | `memory/summarizer.py` | ✅ |
| Repeated-bad detection | `memory/similarity.py` | ✅ |
| BaselineManager | `manager/baseline_manager.py` | ✅ |
| PromptManager | `manager/prompt_manager.py` | ✅ |
| LangGraphManager | `manager/langgraph_manager.py` | ✅ |
| Worker protocol | `worker/base.py` | ✅ |
| ClawWorker (proposal-driven packets) | `worker/claw_worker.py` | ✅ |
| Pending-trial guard | `control_plane/campaign.py` | ✅ |
| Dry-run campaign | `control_plane/campaign.py` | ✅ |
| Real campaign | `control_plane/campaign.py` | ✅ |
| Paper metrics | `evaluation/metrics.py` | ✅ |
| Campaign summary | `evaluation/campaign_summary.py` | ✅ |
| Memory ablation plan | `evaluation/ablations.py` | ✅ |
| Table export | `reporting/export_tables.py` | ✅ |
| Figure CSV export | `reporting/export_figures.py` | ✅ |
| ResNet metric parser | `nodes/resnet_trigger/metric_parser.py` | ✅ |
| Legacy loop adapter | `legacy/claw_code.py` | ✅ |

**What still needs a real run to validate:** the `run_real_campaign()` path with a live Ollama model and the actual ResNet-trigger node. All control-plane, decision, memory, and evaluation paths are verified by smoke tests against mock workers.